# TwoTower Model

Neural two-tower recommender, ranking task, compared against SVD via graded NDCG.

In [1]:
# Imports
import numpy as np
import pandas as pd

from libreco.data import DatasetPure
from libreco.algorithms import TwoTower

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

Instructions for updating:
non-resource variables are not supported in the long term


In [2]:
# Load Data
train = pd.read_csv("train_ratings.csv")
val   = pd.read_csv("val_ratings.csv")
test  = pd.read_csv("test_ratings.csv")

print(f"Train: {len(train):,}  |  Val: {len(val):,}  |  Test: {len(test):,}")
print("Columns:", list(train.columns))

Train: 3,235,873  |  Val: 404,483  |  Test: 404,483
Columns: ['user_id', 'book_id', 'rating']


## 1. Data Preparation

- `LibRecommender` requires columns named `user`, `item`, `label` (user/item must be the first two columns). 
- We rename our columns and set `label = 1` for all interactions — the "all interactions = positive" decision for ranking.

In [3]:
def to_ranking_format(df):
    """Rename to LibRecommender's expected columns; all interactions = positive."""
    out = df.rename(columns={"user_id": "user", "book_id": "item"}).copy()
    out["label"] = 1
    return out[["user", "item", "label"]]   # user/item must be first two columns

train_rk = to_ranking_format(train)
val_rk   = to_ranking_format(val)

print(train_rk.head())
print("\nShape:", train_rk.shape)
print("All labels = 1:", (train_rk['label'] == 1).all())

                               user      item  label
0  8842281e1d1347389f2ab93d60773d4d  23310161      1
1  8842281e1d1347389f2ab93d60773d4d    817720      1
2  8842281e1d1347389f2ab93d60773d4d   1969280      1
3  8842281e1d1347389f2ab93d60773d4d  17290220      1
4  8842281e1d1347389f2ab93d60773d4d   1027760      1

Shape: (3235873, 3)
All labels = 1: True


## 2. Build Dataset and Initialize Model

Build the `LibRecommender` dataset objects, then instantiate TwoTower with `task="ranking"` (the only task it supports). 

In [4]:
# Build LibRecommender dataset objects.
# build_trainset returns (transformed data, data_info); data_info holds
# user/item counts and mappings used throughout training and prediction.
# build_evalset prepares the validation data for per-epoch monitoring.

train_data, data_info = DatasetPure.build_trainset(train_rk)
eval_data = DatasetPure.build_evalset(val_rk)

print(data_info)

n_users: 61078, n_items: 22931, data density: 0.2310 %


In [22]:
# Reset graph (TF1 mode — avoids "Variable already exists" on re-run)
import tensorflow as tf
tf.compat.v1.reset_default_graph()

# FINAL TwoTower model — config locked after manual hyperparameter search.
# Full search recorded in notes_twotower_implementation.md.
twotower = TwoTower(
    task="ranking",                # only task TwoTower supports
    data_info=data_info,
    loss_type="softmax",           # in-batch negatives (Yi et al. 2019); fits top-K retrieval
    embed_size=32,                 # 32 > 64 (64 unstable on sparse data)
    norm_embed=True,               # L2-normalise tower outputs → score is cosine in [-1,1]
    n_epochs=12,                   # early-stopping point: val ndcg plateaus ~ep12 (0.408),
                                   #   train_loss keeps falling after → mild overfit beyond 12
    lr=0.001,                      # 0.001 > 0.003
    batch_size=2048,               # governs # of in-batch negatives under softmax
    num_neg=1,                     # inert under softmax (negatives are in-batch); kept default
    use_bn=True,
    hidden_units=(128, 64, 32),    # architecture of each tower
    temperature=0.1,               # ⭐ decisive. Swept 1.0 / 0.15 / 0.1 / 0.05 → inverted-U,
                                   #   peak at 0.1: ndcg 0.085 / 0.373 / 0.394 / 0.394.
                                   #   Low temp required: norm_embed compresses scores to [-1,1].
    seed=RANDOM_STATE,
)
print("Final TwoTower initialised (n_epochs=12)")

Final TwoTower initialised (n_epochs=12)


## 3. Model Training

Final training run with the locked configuration (see hyperparameter search in notes). 
- `n_epochs=12` chosen as the early-stopping point where validation NDCG plateaus. 
- `eval_user_num=None` evaluates on the full validation set for an accurate per-epoch monitoring curve.

In [23]:
# Final training run (n_epochs=12, locked config).
twotower.fit(
    train_data,
    neg_sampling=True,    # required for ranking with positive-only data
    verbose=2,
    shuffle=True,         # VAL set, monitoring only
    eval_data=eval_data,
    metrics=["loss", "precision", "recall", "ndcg"],
    k=10,
    eval_user_num=None,  # full val set (~4-5 min/epoch).
)

Training start time: 2026-06-09 19:32:47


/Users/jhuang/Library/CloudStorage/GoogleDrive-dimitripanch@gmail.com/My Drive/College/Graduate/NEU/2026 Summer/CS5100 Foundation of AI/Final Project/A-Collaborative-Filtering-Approach-to-Book-Recommendation/venv311/lib/python3.11/site-packages/libreco/layers/dense.py:31: UserWarning: `tf.layers.batch_normalization` is deprecated and will be removed in a future version. Please use `tf.keras.layers.BatchNormalization` instead. In particular, `tf.control_dependencies(tf.GraphKeys.UPDATE_OPS)` should not be used (consult the `tf.keras.layers.BatchNormalization` documentation).
  net = tf.layers.batch_normalization(net, training=is_training)
/Users/jhuang/Library/CloudStorage/GoogleDrive-dimitripanch@gmail.com/My Drive/College/Graduate/NEU/2026 Summer/CS5100 Foundation of AI/Final Project/A-Collaborative-Filtering-Approach-to-Book-Recommendation/venv311/lib/python3.11/site-packages/libreco/layers/dense.py:34: UserWarning: `tf.layers.dense` is deprecated and will be removed in a future vers

total params: 2,718,336 | embedding params: 2,688,768 | network params: 29,568


train: 100%|██████████| 1581/1581 [00:41<00:00, 37.74it/s]


Epoch 1 elapsed: 41.896s
	 train_loss: 7.2643


...

	 eval log_loss: 0.5625
	 eval precision@10: 0.1169
	 eval recall@10: 0.2590
	 eval ndcg@10: 0.4042


train: 100%|██████████| 1581/1581 [00:38<00:00, 41.16it/s]


Epoch 12 elapsed: 38.416s
	 train_loss: 6.2092


eval_listwise: 100%|██████████| 58837/58837 [03:56<00:00, 248.43it/s]


	 eval log_loss: 0.5624
	 eval precision@10: 0.1182
	 eval recall@10: 0.2623
	 eval ndcg@10: 0.4068


## 4. Evaluation Setup — Shared Candidate Pools

For graded-NDCG comparison, every model is scored on the SAME candidate pool
per user: their held-out test items + a fixed sample of non-interacted items.
Built once here, reused by Popularity / TwoTower / SVD.

In [13]:
# Books each user interacted with in TRAIN (to exclude when sampling negatives)
train_items_by_user = train.groupby("user_id")["book_id"].apply(set).to_dict()

# Held-out TEST ratings per user: {user: {book: rating}} — graded relevance source
test_ratings_by_user = (
    test.groupby("user_id")
        .apply(lambda g: dict(zip(g["book_id"], g["rating"])))
        .to_dict()
)

all_items = train["book_id"].unique()

print(f"Users with test data: {len(test_ratings_by_user):,}")
print(f"Total unique items:   {len(all_items):,}")
# sanity check one user
u0 = next(iter(test_ratings_by_user))
print(f"Example user {u0[:8]}... has {len(test_ratings_by_user[u0])} test ratings")

Users with test data: 58,787
Total unique items:   22,931
Example user 00004584... has 4 test ratings


## 4. Evaluation Setup — Shared Candidate Pools

For graded-NDCG comparison, every model is scored on the SAME candidate pool per
user: their held-out test items (graded by true rating) + 100 sampled negatives
drawn from non-interacted items. Built once with a fixed seed so Popularity /
TwoTower / SVD are evaluated on identical candidates — a controlled comparison.
Following the sampled-evaluation protocol of He et al. (2017).

In [25]:
def build_candidate_pools(test_ratings_by_user, train_items_by_user,
                          all_items, n_neg=100, seed=RANDOM_STATE):
    """For each test user: pool = their test items + n_neg sampled negatives.
    Negatives exclude anything the user interacted with in train.
    Built once and reused across all models for a fair comparison."""
    rng = np.random.default_rng(seed)
    all_items_arr = np.asarray(all_items)
    pools = {}

    for user, test_items in test_ratings_by_user.items():
        seen = train_items_by_user.get(user, set())          # exclude train items
        pos_items = set(test_items.keys())                   # held-out test items (graded)
        exclude = seen | pos_items

        # sample negatives not in exclude (oversample then filter to be safe)
        negs = []
        while len(negs) < n_neg:
            cand = rng.choice(all_items_arr, size=n_neg * 2, replace=False)
            negs = [it for it in cand if it not in exclude][:n_neg]

        pools[user] = list(pos_items) + negs                 # full candidate pool

    return pools

# Build once
candidate_pools = build_candidate_pools(
    test_ratings_by_user, train_items_by_user, all_items, n_neg=100
)
print(f"Built candidate pools for {len(candidate_pools):,} users")
u0 = next(iter(candidate_pools))
print(f"Example user pool size: {len(candidate_pools[u0])} "
      f"({len(test_ratings_by_user[u0])} test + {len(candidate_pools[u0]) - len(test_ratings_by_user[u0])} neg)")

Built candidate pools for 58,787 users
Example user pool size: 104 (4 test + 100 neg)


### Graded NDCG Evaluation

A single evaluation function shared by all models. For each user, the model
scores their candidate pool; we compute graded NDCG@K using the true test
ratings as relevance (via sklearn). Only the `score_fn` differs between models —
the candidate pools, ground truth, and metric code are identical.

In [26]:
from sklearn.metrics import ndcg_score

def evaluate_ndcg(score_fn, candidate_pools, test_ratings_by_user, k=10):
    """Mean graded NDCG@k over all users.
    score_fn(user, items) -> list of predicted scores (one per item, ranking only).
    Relevance = true test rating (0 if the item isn't a held-out positive)."""
    ndcgs = []

    for user, items in candidate_pools.items():
        ratings = test_ratings_by_user[user]                 # {book: true rating}
        y_true = [ratings.get(it, 0) for it in items]        # graded relevance (0 = negative)

        # need at least one positive and one non-positive to rank
        if sum(y_true) == 0 or len(set(y_true)) == 1:
            continue

        y_score = score_fn(user, items)                      # model's predicted scores
        ndcgs.append(ndcg_score([y_true], [y_score], k=k))

    return float(np.mean(ndcgs)), len(ndcgs)

## 5. Popularity Baseline

A non-personalized lower bound: score every book by how many users rated it in
the training set (popularity), ignoring who the user is. Any personalized model
that fails to beat this hasn't justified its complexity. This is also the
simplest model, so it validates the evaluation pipeline end-to-end.

In [27]:
# Book popularity = number of training interactions per book.
book_popularity = train["book_id"].value_counts().to_dict()

def popularity_score_fn(user, items):
    """Score = book's training popularity. Same for every user (non-personalized)."""
    return [book_popularity.get(it, 0) for it in items]

pop_ndcg, n_users = evaluate_ndcg(
    popularity_score_fn, candidate_pools, test_ratings_by_user, k=10
)
print(f"Popularity baseline — graded NDCG@10: {pop_ndcg:.4f}  (over {n_users:,} users)")

Popularity baseline — graded NDCG@10: 0.6992  (over 58,787 users)


### Diagnostic — Popularity bias in negative sampling

Positives (user-read books) are far more popular than randomly sampled negatives,
which is why the popularity baseline scores artificially high. Evidence below.

In [30]:
# Evidence of popularity bias: positives (books users actually read) skew popular,
# while random negatives are mostly long-tail. This explains the inflated
# popularity-baseline NDCG. Sampled over 5,000 users.
import numpy as np

pos_pops, neg_pops = [], []
for user, items in list(candidate_pools.items())[:5000]:   # sample 5,000 users
    ratings = test_ratings_by_user[user]
    for it in items:
        pop = book_popularity.get(it, 0)
        if it in ratings:
            pos_pops.append(pop)
        else:
            neg_pops.append(pop)

print(f"Positives — mean popularity: {np.mean(pos_pops):.1f}  (median {np.median(pos_pops):.0f})")
print(f"Negatives — mean popularity: {np.mean(neg_pops):.1f}  (median {np.median(neg_pops):.0f})")
print(f"Positive/negative popularity ratio: {np.mean(pos_pops)/max(np.mean(neg_pops),1):.1f}x")

Positives — mean popularity: 3455.9  (median 757)
Negatives — mean popularity: 131.9  (median 36)
Positive/negative popularity ratio: 26.2x


## 6. TwoTower — Graded NDCG

Score each user's candidate pool with the trained TwoTower model, on the SAME
pools as the popularity baseline. This is the decisive comparison: can TwoTower
beat the popularity baseline (0.6992), i.e. learn signal beyond popularity?

In [29]:
def twotower_score_fn(user, items):
    """Score candidates with the trained TwoTower model.
    predict accepts a single user repeated against the item list.
    cold_start='popular' handles any item the model didn't see in training."""
    users = [user] * len(items)
    return twotower.predict(user=users, item=items, cold_start="popular")

tt_ndcg, n_users = evaluate_ndcg(
    twotower_score_fn, candidate_pools, test_ratings_by_user, k=10
)
print(f"TwoTower — graded NDCG@10: {tt_ndcg:.4f}  (over {n_users:,} users)")
print(f"Popularity baseline:        0.6992")
print(f"Difference:                 {tt_ndcg - 0.6992:+.4f}")

TwoTower — graded NDCG@10: 0.8508  (over 58,787 users)
Popularity baseline:        0.6992
Difference:                 +0.1516
